In [ ]:
# Install necessary libraries
! pip install qiskit-machine-learning

In [8]:
import qiskit
print(qiskit.__version__)


2.0.0


In [16]:
from qiskit.circuit.library import ZZFeatureMap
from qiskit import qasm2
import numpy as np

# Create feature map
feature_map = ZZFeatureMap(feature_dimension=4, reps=2)

# Prepare dummy values (for example, fill with 0.5)
parameter_values = np.full(len(feature_map.parameters), 0.5)

# Bind parameters
bound_circuit = feature_map.assign_parameters(parameter_values)

# Draw the circuit
bound_circuit.draw(output='mpl')

# Now export to QASM2
qasm_text = qasm2.dumps(bound_circuit)

# Save to a file
with open('feature_map.qasm', 'w') as f:
    f.write(qasm_text)


In [ ]:
# Iris_dataset_Quantum_approach

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
import numpy as np

from qiskit.circuit.library import ZZFeatureMap
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

# 1. Load Iris dataset
iris = load_iris()
X = iris.data
y = iris.target

# Binary classification (class 0 vs 1 only)
X = X[y != 2]
y = y[y != 2]

# Scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# 2. Feature map
feature_map = ZZFeatureMap(feature_dimension=X_train.shape[1], reps=2)

# 3. Feature extraction function (No Sampler used!)
def quantum_feature(x):
    param_dict = dict(zip(feature_map.parameters, x))
    qc = feature_map.assign_parameters(param_dict)
    statevector = Statevector.from_instruction(qc)
    return np.real(statevector.data)  # Only real part

# 4. Extract quantum features
X_train_features = np.array([quantum_feature(x) for x in X_train])
X_test_features = np.array([quantum_feature(x) for x in X_test])

# 5. Train classical SVM
classifier = SVC()
classifier.fit(X_train_features, y_train)

# 6. Predict and Evaluate
y_pred = classifier.predict(X_test_features)

from sklearn.metrics import accuracy_score
print("Accuracy:", accuracy_score(y_test, y_pred))


Accuracy: 0.65
